# Dataset Loaders
This notebook builds the full 500‑email dataset from the sources listed in the PDF.
We keep the same target schema as the demo notebook.

In [1]:
from __future__ import annotations

import re
from typing import List, Dict
from pathlib import Path

import pandas as pd

SCHEMA_COLUMNS = [
    "email_id" ,
    "source" ,
    "actual_class" ,
    "sender_address" ,
    "subject_line" ,
    "body_content" ,
    "extracted_links" ,
]

URL_REGEX = re.compile(r"https?://[^\s<>]+", re.IGNORECASE)

def extract_links(text: str) -> List[str]:
    if not text:
        return []
    return URL_REGEX.findall(text)

def normalize_record(
    email_id: str,
    source: str,
    actual_class: int,
    sender_address: str,
    subject_line: str,
    body_content: str,
) -> Dict[str, object]:
    body_content = body_content or ""
    subject_line = subject_line or ""
    extracted_links = extract_links(body_content)
    return {
        "email_id": email_id,
        "source": source,
        "actual_class": int(actual_class),
        "sender_address": sender_address or "" ,
        "subject_line": subject_line,
        "body_content": body_content,
        "extracted_links": extracted_links,
    }

def validate_schema(df: pd.DataFrame) -> None:
    missing = [c for c in SCHEMA_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

## Local Data Configuration
All datasets are loaded from local CSV files (no Kaggle/URL downloads).

In [2]:
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Local data file paths (all stored in data/ directory)
# Benign emails (SpamAssassin or generic emails)
BENIGN_PATH = DATA_DIR / "emails.csv"  # Generic benign emails dataset
# Alternative if preferred:
# SPAMASSASSIN_PATH = DATA_DIR / "spamassassin_ham_100.csv"

# Phishing sources
PHISHBOWL_PATH = DATA_DIR / "cornell_phishbowl_data.csv"
LEGACY_LLM_PHISHING_PATH = DATA_DIR / "legacy_llm_phishing.csv"
HYBRID_PATH = DATA_DIR / "vtriad_hybrid_diverse.csv"

# Verify all required files exist
REQUIRED_FILES = {
    "Benign Emails": BENIGN_PATH,
    "Phishbowl": PHISHBOWL_PATH,
    "Legacy LLM Phishing": LEGACY_LLM_PHISHING_PATH,
    "Hybrid": HYBRID_PATH,
}

print("=== LOCAL DATA FILES ===")
missing_files = []
for name, path in REQUIRED_FILES.items():
    exists = path.exists()
    status = "✓" if exists else "✗"
    print(f"{status} {name}: {path.name} ({path.stat().st_size / 1024:.1f} KB)" if exists else f"{status} {name}: {path} (MISSING)")
    if not exists:
        missing_files.append(f"{name} ({path})")

if missing_files:
    print(f"\n⚠ Missing files:\n  - " + "\n  - ".join(missing_files))
    print("Make sure all CSV files are in the data/ directory.")
else:
    print("\n✓ All required data files found locally!")

=== LOCAL DATA FILES ===
✓ Benign Emails: emails.csv (1392697.5 KB)
✓ Phishbowl: cornell_phishbowl_data.csv (439.0 KB)
✓ Legacy LLM Phishing: legacy_llm_phishing.csv (24.2 KB)
✓ Hybrid: vtriad_hybrid_diverse.csv (11.6 KB)

✓ All required data files found locally!


In [ ]:
def load_benign_emails(path: Path) -> pd.DataFrame:
    """
    Load benign emails dataset (columns: id, subject, sender, body).
    Flexible to handle emails.csv or spamassassin CSV format.
    """
    df = pd.read_csv(path, nrows=100)  # Load first 100 for balance
    rows = []
    for i, row in df.iterrows():
        # Try multiple column name variants
        sender = row.get('sender', row.get('from', row.get('email_from', '')))
        subject = row.get('subject', row.get('Subject', ''))
        body = row.get('body', row.get('Body', row.get('email_body', '')))
        
        rows.append(normalize_record(
            email_id=f"BEN_{i+1}",
            source='Benign_Emails',
            actual_class=0,  # benign
            sender_address=sender if sender and str(sender).lower() != 'nan' else '',
            subject_line=subject if subject and str(subject).lower() != 'nan' else '',
            body_content=body if body and str(body).lower() != 'nan' else '',
        ))
    return pd.DataFrame(rows)


def load_phishbowl(path: Path) -> pd.DataFrame:
    """Load Phishbowl phishing dataset (columns: title, email_message)."""
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"PB_{i+1}",
            source='Phishbowl',
            actual_class=1,  # phishing
            sender_address=row.get('Unnamed: 0', '') if 'Unnamed: 0' in df.columns else '',
            subject_line=row.get('title', ''),
            body_content=row.get('email_message', ''),
        ))
    return pd.DataFrame(rows)


def load_legacy_llm_phishing(path: Path) -> pd.DataFrame:
    """Load legacy LLM phishing dataset (generated format: subject, sender, body)."""
    if not path.exists():
        return pd.DataFrame(columns=SCHEMA_COLUMNS)
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"LLM_{i+1}",
            source='Legacy_LLM_Phishing',
            actual_class=1,  # phishing
            sender_address=row.get('sender', ''),
            subject_line=row.get('subject', ''),
            body_content=row.get('body', ''),
        ))
    return pd.DataFrame(rows)


def load_hybrid(path: Path) -> pd.DataFrame:
    """Load hybrid phishing dataset (generated format: subject, sender, body)."""
    if not path.exists():
        return pd.DataFrame(columns=SCHEMA_COLUMNS)
    df = pd.read_csv(path)
    rows = []
    for i, row in df.iterrows():
        rows.append(normalize_record(
            email_id=f"HYB_{i+1}",
            source='Self_Generated_Hybrid',
            actual_class=1,  # phishing
            sender_address=row.get('sender', ''),
            subject_line=row.get('subject', ''),
            body_content=row.get('body', ''),
        ))
    return pd.DataFrame(rows)


print("✓ All loader functions defined")

✓ Benign email loader defined


In [4]:
def build_master_dataset() -> pd.DataFrame:
    """
    Build master dataset from LOCAL CSV files only.
    No Kaggle API, no URL downloads - all data is kept local.
    """
    
    print("=== BUILDING MASTER DATASET (LOCAL FILES) ===\n")
    
    frames = []
    
    # Load Benign Emails
    if BENIGN_PATH.exists():
        print(f"Loading Benign from: {BENIGN_PATH.name}")
        benign_df = load_benign_emails(BENIGN_PATH)
        print(f"  ✓ Loaded {len(benign_df)} emails, body avg: {benign_df['body_content'].str.len().mean():.0f} chars\n")
        frames.append(benign_df)
    else:
        print(f"  ✗ Benign not found: {BENIGN_PATH}\n")
    
    # Load Phishbowl (real phishing)
    if PHISHBOWL_PATH.exists():
        print(f"Loading Phishbowl from: {PHISHBOWL_PATH.name}")
        pb_df = load_phishbowl(PHISHBOWL_PATH)
        print(f"  ✓ Loaded {len(pb_df)} emails, body avg: {pb_df['body_content'].str.len().mean():.0f} chars\n")
        frames.append(pb_df)
    else:
        print(f"  ✗ Phishbowl not found: {PHISHBOWL_PATH}\n")
    
    # Load Legacy LLM Phishing
    if LEGACY_LLM_PHISHING_PATH.exists():
        print(f"Loading Legacy LLM Phishing from: {LEGACY_LLM_PHISHING_PATH.name}")
        llm_df = load_legacy_llm_phishing(LEGACY_LLM_PHISHING_PATH)
        print(f"  ✓ Loaded {len(llm_df)} emails, body avg: {llm_df['body_content'].str.len().mean():.0f} chars\n")
        frames.append(llm_df)
    else:
        print(f"  ✗ Legacy LLM not found: {LEGACY_LLM_PHISHING_PATH}\n")
    
    # Load Hybrid (synthetic)
    if HYBRID_PATH.exists():
        print(f"Loading Hybrid from: {HYBRID_PATH.name}")
        hyb_df = load_hybrid(HYBRID_PATH)
        print(f"  ✓ Loaded {len(hyb_df)} emails, body avg: {hyb_df['body_content'].str.len().mean():.0f} chars\n")
        frames.append(hyb_df)
    else:
        print(f"  ✗ Hybrid not found: {HYBRID_PATH}\n")
    
    # Combine all frames
    if not frames:
        raise ValueError("No datasets loaded! Check file paths.")
    
    master = pd.concat(frames, ignore_index=True)
    validate_schema(master)
    
    print(f"=== MASTER DATASET BUILT ===")
    print(f"Total rows: {len(master)}")
    print(f"\nBreakdown by source:")
    print(master['source'].value_counts())
    print(f"\nBreakdown by class:")
    print(master['actual_class'].value_counts())
    print(f"\nAverage body content length by source:")
    print(master.groupby('source')['body_content'].apply(lambda x: x.str.len().mean()).round(0))
    
    return master


# Build the master dataset
print("Loading datasets from local CSV files...\n")
master_df = build_master_dataset()

# Save for downstream notebooks
output_path = DATA_DIR / "master_legacy.csv"
master_df.to_csv(output_path, index=False)
print(f"\n✓ Saved to: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")

Loading datasets from local CSV files...

=== BUILDING MASTER DATASET (LOCAL FILES) ===

Loading Benign from: emails.csv
  ✓ Loaded 100 emails, body avg: 0 chars

Loading Phishbowl from: cornell_phishbowl_data.csv


NameError: name 'load_phishbowl' is not defined

In [20]:
print(f"Total rows: {len(master_df)}")
print(f"\nBreakdown by source:")
print(master_df['source'].value_counts())
print(f"\nBreakdown by class:")
print(master_df['actual_class'].value_counts())
print(f"\nSample with content:")
print(master_df[master_df['body_content'].str.len() > 0].head(2))

Total rows: 488

Breakdown by source:
source
Phishbowl                240
SpamAssassin             100
Legacy_LLM_Phishing      100
Self_Generated_Hybrid     48
Name: count, dtype: int64

Breakdown by class:
actual_class
1    388
0    100
Name: count, dtype: int64

Sample with content:
    email_id               source  actual_class              sender_address  \
340    LLM_1  Legacy_LLM_Phishing             1     it-security@example.com   
341    LLM_2  Legacy_LLM_Phishing             1  payroll-update@example.com   

                              subject_line  \
340  Action Required: Access Review Window   
341   Important: Compliance Acknowledgment   

                                          body_content extracted_links  
340  Dear Jordan,\n\nAs part of compliance at Cedar...              []  
341  Hello Casey,\n\nWe are finalizing the quarterl...              []  
